# 01 · 训练/验证/测试划分与交叉验证 Train/Val/Test Split & Cross-Validation

> **能力标签**：SH7（经典机器学习 / Classical ML）

## 目标 Objectives

完成本课后，你应该能够：

1. 理解**训练集 / 验证集 / 测试集**（train / validation / test set）的作用与划分原则。
2. 掌握 **k 折交叉验证**（k-fold cross-validation）与 **StratifiedKFold** 的思想与实现。
3. 使用 `sklearn.model_selection.cross_val_score` / `StratifiedKFold` 评估模型泛化性能。
4. 理解数据泄露（data leakage）的常见来源及规避方法。

> 对应能力：**SH7**

## 概念讲解 Concepts

### 为何要划分数据集？

机器学习的核心目标是**泛化**（generalization）——在未见过的样本上表现良好。
若将同一批数据既用于训练又用于评估，评估结果会过于乐观（即**过拟合**检测失灵）。

| 子集 | 用途 |
|------|------|
| **训练集** (train) | 拟合模型参数 |
| **验证集** (validation) | 调整超参数、模型选择 |
| **测试集** (test) | 最终、只读一次的性能报告 |

典型比例：**60/20/20** 或 **70/15/15**。

---

### k 折交叉验证 k-Fold Cross-Validation

数据量不足时，单次划分方差较大。k 折 CV 将数据分成 $k$ 份，轮流用每一份作验证集：

$$\text{CV score} = \frac{1}{k}\sum_{i=1}^{k} \text{score}^{(i)}$$

- **StratifiedKFold**：保持每折类别比例，适合不平衡分类。
- **GroupKFold**：同一样本（如同一病人）的所有记录在同一折中，防止泄露。

---

### 数据泄露 Data Leakage

在训练前用**全部数据**（包含测试集）做特征工程（如 z-score、PCA），会导致泄露。
正确做法：**在 Pipeline 内**做预处理，仅在训练折上拟合变换器（transformer）。

## 示例 Worked Example — `kfold_indices`

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

# ── 手写 kfold_indices（镜像题包 w5-cv）──────────────────────────────────────
def kfold_indices(n: int, k: int, seed: int = 0):
    """返回 k 折的 (train_idx, test_idx) 列表，测试折两两不相交且并集为全部样本。"""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    folds = np.array_split(idx, k)
    out = []
    for i in range(k):
        test  = np.sort(folds[i])
        train = np.sort(np.concatenate([folds[j] for j in range(k) if j != i]))
        out.append((train, test))
    return out

# ── 验证：测试折不相交、并集 = 全集 ─────────────────────────────────────────
n, k = 100, 5
splits = kfold_indices(n, k, seed=42)
all_test = np.concatenate([te for _, te in splits])
print(f"k={k} 折 | 各测试折大小: {[len(te) for _, te in splits]}")
print(f"测试折并集大小 = {len(np.unique(all_test))}（应为 {n}）")

# ── sklearn StratifiedKFold + cross_val_score ─────────────────────────────
X, y = make_classification(n_samples=300, n_features=10, n_informative=5,
                            random_state=42)
clf = LogisticRegression(max_iter=500)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(clf, X, y, cv=skf, scoring="roc_auc")
print(f"\n5-Fold CV ROC-AUC: {scores.round(3)}")
print(f"均值 ± 标准差: {scores.mean():.3f} ± {scores.std():.3f}")

## 动手 Exercise

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

# 练习：比较 k=3, 5, 10 折的 CV 均值与方差
X, y = make_classification(n_samples=500, n_features=15, n_informative=6,
                            random_state=7)
clf = LogisticRegression(max_iter=500, random_state=0)

print(f"{'k':>3}  {'CV AUC mean':>12}  {'CV AUC std':>11}")
print("-" * 32)
for k in [3, 5, 10]:
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=0)
    sc = cross_val_score(clf, X, y, cv=skf, scoring="roc_auc")
    print(f"{k:>3}  {sc.mean():>12.4f}  {sc.std():>11.4f}")

# 结论观察：k 越大，均值更稳定但计算代价增加
print("\n观察：随着 k 增大，方差（std）通常减小，但计算量线性增加。")

## 延伸阅读 Further Reading

1. **sklearn 交叉验证指南**：<https://scikit-learn.org/stable/modules/cross_validation.html>
2. **《The Elements of Statistical Learning》** Hastie et al. — 第 7 章（模型评估与选择）
3. **Cawley & Talbot (2010)**："On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation" — JMLR。
4. **sklearn Pipeline 文档**：<https://scikit-learn.org/stable/modules/pipeline.html>

## 关联练习 Related Assignments

```bash
python check.py 01-cv
```

> 实现 `kfold_indices(n, k, seed)`，使测试折两两不相交且并集为全部样本索引。

> 能力标签：**SH7** · threshold ≥ 0.7